# 02 · Synthetic pipeline demonstration

**This notebook does not reproduce the paper's results.** It runs the ABTC pipeline (10-fold stratified CV, 3-fold isotonic calibration inside each training split, Path A / Path B selection) on synthetic data generated with `sklearn.datasets.make_classification`, to show how the pieces fit together.

The paper's main results were produced with `experiments/ABTC_V2_Final.py` (one dataset per run, see the repository README); the revision experiments with `ablation/ABTC_Ablation_Final.ipynb`.


In [ ]:
# Install dependencies
%pip install -e . -q

## Configuration

In [ ]:
import pandas as pd
import numpy as np
import pathlib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV

# ABTC imports
from abtc import (
    isotonic_calibrator,
    alpha_adjust_posterior,
    joint_alpha_tau_search,
    alpha_search_analytical_tau,
    tau_cost,
    apply_threshold,
    get_search_config
)
from abtc.thresholding import sweep_metric_threshold
from abtc.metrics import fbeta, gmean, mcc, balanced_acc, pr_auc

print("ABTC library imported successfully.")

In [ ]:
# =============================================================================
# EXPERIMENT CONFIGURATION
# =============================================================================

RANDOM_STATE = 42
KFOLD = 10
N_BOOTSTRAP = 1000

# Primary metric (change as needed)
PRIMARY_METRIC = 'F2'  # Options: 'F1', 'F2', 'G-Mean', 'MCC', 'BalancedAcc', 'PR-AUC', 'cost'

# Cost configuration (for Path B)
COST_FP = 1.0
COST_FN = 10.0

# =============================================================================
# ABTC SINGLE-LAYER SEARCH CONFIGURATION
# =============================================================================

N_ALPHA = 27      # Number of α values
N_TAU = 20        # Number of τ values
ALPHA_RANGE = (0.1, 3.0)
TAU_RANGE = (0.01, 0.99)

# Display configuration
print("="*70)
print("ABTC V1 - SINGLE-LAYER SEARCH CONFIGURATION")
print("="*70)
print(f"\nExperiment Settings:")
print(f"  Random State:    {RANDOM_STATE}")
print(f"  K-Fold CV:       {KFOLD}")
print(f"  Primary Metric:  {PRIMARY_METRIC}")
if PRIMARY_METRIC == 'cost':
    print(f"  Cost Config:     C_FP={COST_FP}, C_FN={COST_FN}")

print(f"\nSearch Grid:")
print(f"  α: {N_ALPHA} values in [{ALPHA_RANGE[0]}, {ALPHA_RANGE[1]}], Δα ≈ {(ALPHA_RANGE[1]-ALPHA_RANGE[0])/(N_ALPHA-1):.3f}")
print(f"  τ: {N_TAU} values in [{TAU_RANGE[0]}, {TAU_RANGE[1]}], Δτ ≈ {(TAU_RANGE[1]-TAU_RANGE[0])/(N_TAU-1):.3f}")
print(f"\nPath A evaluations: {N_ALPHA} × {N_TAU} = {N_ALPHA * N_TAU}")
print(f"Path B evaluations: {N_ALPHA} (τ analytical)")
print("="*70)

## Load Configuration Files

In [ ]:
ROOT = pathlib.Path("..")

# Synthetic demonstration: three synthetic datasets (no real data are read).
SRC = pd.DataFrame({'dataset_name': ['Synthetic 1', 'Synthetic 2', 'Synthetic 3'],
                    'domain': ['Synthetic'] * 3})

# Primary-metric policy (rows for the synthetic names fall back to PRIMARY_METRIC)
try:
    policy = pd.read_csv(ROOT / "config" / "metrics_policy.csv")
except FileNotFoundError:
    policy = pd.DataFrame(columns=['dataset_name', 'primary_metric', 'beta', 'cost_ratio'])


## Utility Functions

In [ ]:
def metric_from_row(row):
    """Determine primary metric for a dataset."""
    # Dataset-level override (from CSV)
    r = policy[policy["dataset_name"] == row["dataset_name"]]
    if len(r):
        m = str(r.iloc[0]["primary_metric"]).strip()
        b = r.iloc[0]["beta"]
        if m:
            return m, (int(b) if str(b).strip().isdigit() else None)
    
    # Inline per-row (optional)
    m = str(row.get("primary_metric", "")).strip()
    if m:
        b = row.get("beta", None)
        return m, (int(b) if str(b).strip().isdigit() else None)
    
    # Domain default
    dom = str(row.get("domain", "")).strip()
    if dom in defaults and "primary_metric" in defaults[dom]:
        dm = defaults[dom]["primary_metric"]
        db = defaults[dom].get("beta", None)
        return dm, db
    
    # Fallback heuristic by imbalance
    try:
        pr = float(row.get("pos_rate", np.nan))
    except:
        pr = np.nan
    if pd.notna(pr) and pr <= 0.1:
        return "F2", 2
    return "F1", 1


def metric_callable(name, beta=None):
    """Get metric function from name."""
    name = name.lower()
    if name == "f2":
        return lambda y_true, y_pred: fbeta(y_true, y_pred, 2.0)
    if name == "f1":
        return lambda y_true, y_pred: fbeta(y_true, y_pred, 1.0)
    if name in ("g-mean", "gmean"):
        return gmean
    if name == "mcc":
        return mcc
    if name in ("balancedacc", "balanced accuracy", "balacc"):
        return balanced_acc
    if name in ("pr-auc", "prauc", "ap"):
        return lambda y_true, y_score: pr_auc(y_true, y_score)
    # Default to F2
    return lambda y_true, y_pred: fbeta(y_true, y_pred, 2.0)


def cost_metric(y_true, y_pred, cost_fp=COST_FP, cost_fn=COST_FN):
    """Calculate total misclassification cost."""
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return cost_fp * fp + cost_fn * fn


def evaluate_all_metrics(y_true, y_pred, y_proba):
    """Compute all evaluation metrics."""
    return {
        'F1': fbeta(y_true, y_pred, 1.0),
        'F2': fbeta(y_true, y_pred, 2.0),
        'G-Mean': gmean(y_true, y_pred),
        'MCC': mcc(y_true, y_pred),
        'BalancedAcc': balanced_acc(y_true, y_pred),
        'PR-AUC': pr_auc(y_true, y_proba),
        'cost': cost_metric(y_true, y_pred)
    }

print("Utility functions defined.")

## ABTC Classifier Class (Single-Layer Search)

In [ ]:
class ABTCClassifier:
    """
    Adaptive Bayesian Thresholding Classifier (ABTC)
    ================================================
    
    Single-Layer Search Architecture:
    - Path A: Joint (α, τ) grid search (27×20 = 540 evaluations)
    - Path B: α search with analytical τ* (27 evaluations)
    """
    
    def __init__(self,
                 base_model=None,
                 calibration_method='isotonic',
                 objective='F1',
                 cost_fp=1.0,
                 cost_fn=10.0,
                 n_alpha=27,
                 n_tau=20,
                 alpha_range=(0.1, 3.0),
                 tau_range=(0.01, 0.99),
                 verbose=True):
        
        self.base_model = base_model or LogisticRegression(solver='liblinear', max_iter=1000)
        self.calibration_method = calibration_method
        self.objective = objective
        self.cost_fp = cost_fp
        self.cost_fn = cost_fn
        self.n_alpha = n_alpha
        self.n_tau = n_tau
        self.alpha_range = alpha_range
        self.tau_range = tau_range
        self.verbose = verbose
        
        # Learned parameters
        self.pi_ = None
        self.calibrated = None
        self.best_alpha = None
        self.best_tau = None
        self.optimization_path = None
        self.n_evaluations_ = 0
    
    def fit(self, X, y):
        """Step 1: Train and calibrate base model."""
        self.pi_ = np.mean(y)
        self.calibrated = CalibratedClassifierCV(
            self.base_model, cv=3, method=self.calibration_method
        )
        self.calibrated.fit(X, y)
        return self
    
    def _raw_proba(self, X):
        """Get calibrated probabilities."""
        return self.calibrated.predict_proba(X)[:, 1]
    
    def _get_metric_fn(self):
        """Get metric function for optimization."""
        if self.objective == 'cost':
            return lambda yt, yp: -cost_metric(yt, yp, self.cost_fp, self.cost_fn)
        return metric_callable(self.objective)
    
    def select_threshold(self, X_val, y_val):
        """Step 3: Single-layer search for optimal (α*, τ*)."""
        p_raw = self._raw_proba(X_val)
        metric_fn = self._get_metric_fn()
        
        if self.objective == 'cost':
            # Path B: α search with analytical τ*
            self.optimization_path = 'Path B'
            self.best_alpha, self.best_tau, _, self.n_evaluations_ = alpha_search_analytical_tau(
                y_val=y_val,
                p_raw=p_raw,
                pi=self.pi_,
                adjust_fn=alpha_adjust_posterior,
                cost_fn=self.cost_fn,
                cost_fp=self.cost_fp,
                metric_fn=metric_fn,
                n_alpha=self.n_alpha,
                alpha_range=self.alpha_range
            )
        else:
            # Path A: Joint (α, τ) grid search
            self.optimization_path = 'Path A'
            self.best_alpha, self.best_tau, _, self.n_evaluations_ = joint_alpha_tau_search(
                y_val=y_val,
                p_raw=p_raw,
                pi=self.pi_,
                adjust_fn=alpha_adjust_posterior,
                metric_fn=metric_fn,
                n_alpha=self.n_alpha,
                n_tau=self.n_tau,
                alpha_range=self.alpha_range,
                tau_range=self.tau_range
            )
        
        if self.verbose:
            print(f"  [ABTC {self.optimization_path}] α*={self.best_alpha:.4f}, τ*={self.best_tau:.4f}, evals={self.n_evaluations_}")
    
    def predict_proba(self, X):
        """Get adjusted posterior probabilities."""
        p_raw = self._raw_proba(X)
        return alpha_adjust_posterior(p_raw, self.best_alpha, self.pi_)
    
    def predict(self, X):
        """Final prediction: ŷ = 1 if p_adj ≥ τ*, else 0."""
        return (self.predict_proba(X) >= self.best_tau).astype(int)

print("ABTCClassifier defined (Single-Layer Search).")

## Run Experiments

In [ ]:
# Output directory
OUT = ROOT / "results" / "stats"
OUT.mkdir(parents=True, exist_ok=True)

rows = []

for i, row in SRC.iterrows():
    name = row["dataset_name"]
    dom = row.get("domain", "Unknown")
    
    print(f"\n{'='*60}")
    print(f"Processing: {name} (Domain: {dom})")
    print(f"{'='*60}")
    
    # Synthetic data (demonstration only)
    from sklearn.datasets import make_classification
    X, y = make_classification(
        n_samples=4000, n_features=20, n_informative=10,
        weights=[0.9, 0.1], flip_y=0.01, random_state=i
    )
    
    # Split data
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
    )
    
    # Further split training for validation
    Xtr_inner, Xval, ytr_inner, yval = train_test_split(
        Xtr, ytr, test_size=0.2, stratify=ytr, random_state=RANDOM_STATE
    )
    
    # Scale
    scaler = StandardScaler().fit(Xtr_inner)
    Xtr_s = scaler.transform(Xtr_inner)
    Xval_s = scaler.transform(Xval)
    Xte_s = scaler.transform(Xte)
    
    print(f"  Samples: train={len(ytr_inner)}, val={len(yval)}, test={len(yte)}")
    print(f"  Positive rate: {np.mean(ytr_inner)*100:.1f}%")
    
    # Get metric for this dataset
    metric_name, beta = metric_from_row(row)
    print(f"  Primary metric: {metric_name}")
    
    # Create and train ABTC classifier
    abtc = ABTCClassifier(
        base_model=LogisticRegression(solver='liblinear', max_iter=1000),
        calibration_method='isotonic',
        objective=metric_name if metric_name != 'cost' else 'cost',
        cost_fp=COST_FP,
        cost_fn=COST_FN,
        n_alpha=N_ALPHA,
        n_tau=N_TAU,
        alpha_range=ALPHA_RANGE,
        tau_range=TAU_RANGE,
        verbose=True
    )
    
    # Fit and select threshold
    abtc.fit(Xtr_s, ytr_inner)
    abtc.select_threshold(Xval_s, yval)
    
    # Predict on test set
    p_adj = abtc.predict_proba(Xte_s)
    y_hat = abtc.predict(Xte_s)
    
    # Evaluate
    metrics = evaluate_all_metrics(yte, y_hat, p_adj)
    
    res = {
        'domain': dom,
        'dataset': name,
        'primary_metric': metric_name,
        'beta': beta or "",
        'alpha': abtc.best_alpha,
        'tau': abtc.best_tau,
        'path': abtc.optimization_path,
        'n_evals': abtc.n_evaluations_,
        'pos_rate': row.get("pos_rate", np.mean(y)),
        **metrics
    }
    rows.append(res)
    
    # Save individual result
    pd.DataFrame([res]).to_csv(OUT / f"{str(name).replace(' ', '_')}.csv", index=False)
    
    print(f"  Results: F1={metrics['F1']:.4f}, F2={metrics['F2']:.4f}, G-Mean={metrics['G-Mean']:.4f}")

# Save summary
df_summary = pd.DataFrame(rows)
df_summary.to_csv(OUT / "summary.csv", index=False)
print(f"\n{'='*60}")
print(f"Wrote {len(rows)} result files to {OUT}")
print(f"{'='*60}")

## Results Summary

In [ ]:
# Display summary
print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)

display_cols = ['dataset', 'path', 'alpha', 'tau', 'n_evals', 'F1', 'F2', 'G-Mean', 'MCC']
print(df_summary[display_cols].to_string(index=False))

print("\n" + "="*80)
print("ABTC CONFIGURATION SUMMARY")
print("="*80)
print(f"Search Architecture: Single-Layer Grid Search")
print(f"Path A (Metric):     {N_ALPHA} × {N_TAU} = {N_ALPHA * N_TAU} evaluations")
print(f"Path B (Cost):       {N_ALPHA} evaluations (τ analytical)")
print(f"α range:             [{ALPHA_RANGE[0]}, {ALPHA_RANGE[1]}]")
print(f"τ range:             [{TAU_RANGE[0]}, {TAU_RANGE[1]}]")
print("="*80)

## Summary

This notebook demonstrates the ABTC pipeline on synthetic data using:

### Single-Layer Search Architecture

| Component | Configuration |
|-----------|---------------|
| α grid | 27 values in [0.1, 3.0] |
| τ grid | 20 values in [0.01, 0.99] |
| Path A | 27 × 20 = 540 evaluations |
| Path B | 27 evaluations |
| Calibration | Isotonic (3-fold CV) |

### Output Files

- `results/stats/summary.csv` - All datasets summary
- `results/stats/<dataset>.csv` - Individual dataset results